# Chọn model local cho một bài toán tiếng Việt

Bạn là AI Engineer được giao chọn model chạy local cho tính năng "tự gắn chủ đề cho phản hồi của sinh viên"
(UIT-VSFC, 4 chủ đề). Ứng viên: `Qwen/Qwen2.5-0.5B-Instruct` (Hugging Face, CPU), `qwen2.5:3b` và `qwen2.5:7b`
(Ollama), tuỳ chọn thêm model khác bạn có. Bạn dựng một benchmark nhỏ nhưng trung thực: cùng mẫu, cùng prompt,
cùng cách chấm, đo chất lượng **và** chi phí (độ trễ, bộ nhớ), rồi viết `REPORT.md` khuyến nghị một model.

**Cách làm bài**
- Bốn hàm trong cell chấm điểm có test. Phần thí nghiệm (cell `explore`) là của bạn: khung dưới đây chỉ gợi ý.
- Cần Ollama với `qwen2.5:3b`, `qwen2.5:7b` và bộ `uit-vsfc`. Lần đầu tải `Qwen2.5-0.5B-Instruct` khoảng 1 GB.
- Server Ollama có thể dùng chung: cache kết quả từng câu ra tệp để chạy lại không gọi model lần nữa.
- Chấm: `uv run pytest tracks/ai-engineer/modules/09-llms-and-open-models/exercises/advanced/01-model-selection-benchmark -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import re
import unicodedata
from collections.abc import Sequence

import numpy as np
import pandas as pd

LABELS = ["giảng viên", "chương trình đào tạo", "cơ sở vật chất", "khác"]   # đúng thứ tự mã topic 0..3

## 1. Bốn hàm được chấm

- `stratified_sample(df, label_col, n_per_class, seed) -> DataFrame`: với từng giá trị nhãn theo thứ tự **tăng
  dần**, lấy `group.sample(n=min(n_per_class, len(group)), random_state=seed)`; nối theo thứ tự đó, giữ index gốc.
- `parse_label(reply, labels) -> str | None`: đọc nhãn từ câu trả lời tự do của model. Chuẩn hoá NFC, chữ thường,
  bỏ ở hai đầu mọi khoảng trắng và các ký tự `.` `"` `'` `*` `:` cùng dấu backtick. Trùng khớp nguyên văn một nhãn thì trả nhãn đó (dạng gốc trong
  `labels`). Không thì tìm các nhãn xuất hiện như **cụm từ nguyên vẹn** (không dính chữ cái hay chữ số hai bên:
  "khác" không khớp trong "khách"); đúng một nhãn thì trả nó, không có hoặc nhiều hơn một thì `None`.
- `macro_f1(y_true, y_pred, labels) -> float`: trung bình F1 của từng nhãn trong `labels`; dự đoán `None` là sai
  (tính vào FN của nhãn đúng, không là FP của nhãn nào); nhãn không có TP thì F1 bằng 0.
- `latency_summary(seconds) -> {"p50", "p95", "mean"}`: percentile nội suy tuyến tính (mặc định của
  `np.percentile`), số `float`.

In [ ]:
def stratified_sample(df: pd.DataFrame, label_col: str, n_per_class: int, seed: int) -> pd.DataFrame:
    """Lấy tối đa n_per_class dòng mỗi nhãn (nhãn tăng dần), random_state=seed, giữ index gốc."""
    # TODO
    raise NotImplementedError


def parse_label(reply: str, labels: Sequence[str]) -> str | None:
    """Nhãn đọc được từ câu trả lời, hoặc None nếu không có hoặc mơ hồ."""
    # TODO
    raise NotImplementedError


def macro_f1(y_true: Sequence[str], y_pred: Sequence[str | None], labels: Sequence[str]) -> float:
    """Macro-F1 trên `labels`; None là dự đoán sai."""
    # TODO
    raise NotImplementedError


def latency_summary(seconds: Sequence[float]) -> dict[str, float]:
    """p50, p95, mean của danh sách độ trễ (giây)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dữ liệu: 30 câu mỗi chủ đề từ tập test UIT-VSFC, seed 0 (120 câu)
import json
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

VSFC = ensure("uit-vsfc")
test = pd.read_parquet(VSFC / "default" / "test" / "0000.parquet")
sample = stratified_sample(test, "topic", 30, seed=0)
y_true = [LABELS[t] for t in sample.topic]
print(len(sample), sample.topic.value_counts().sort_index().tolist())
display(sample.head())

In [ ]:
# explore: TODO prompt và chạy các model qua Ollama, cache kết quả từng câu
# Gợi ý: một SYSTEM prompt mô tả 4 nhãn, temperature 0, seed cố định, num_predict nhỏ. Mỗi câu ghi
# {"reply", "seconds", "eval_count", ...} vào một tệp JSON (vd. data/raw/uit-vsfc/cache/bench-<model>.json).
# Sau mỗi model, đọc /api/ps để ghi model chiếm bao nhiêu bộ nhớ và bao nhiêu nằm trên GPU.

In [ ]:
# explore: TODO chạy Qwen/Qwen2.5-0.5B-Instruct bằng transformers trên CPU, cùng prompt (qua apply_chat_template)

In [ ]:
# explore: TODO bảng so sánh: macro-F1, accuracy, số câu trả lời không đọc được nhãn, p50/p95 độ trễ, token/s,
# bộ nhớ; confusion matrix của từng model (pd.crosstab)

## 2. Báo cáo

Viết `REPORT.md` (cùng thư mục) theo khung có sẵn, rồi làm phần trắc nghiệm của bài trong app.